In [1]:
from typing_extensions import TypedDict, Literal, Annotated, Dict 
from langgraph.graph import StateGraph, START, END
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.types import Command, interrupt
from langchain_ollama import ChatOllama
from langchain_core.messages import AIMessage, ToolMessage, HumanMessage
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
from pathlib import Path
import os
import json
import subprocess

In [2]:
MAX_ITER = 5

WORKDIR = Path.cwd()
DENY_LIST = [
    "rm -rf /", "sudo", "shutdown", "reboot",
    "mkfs", "dd if=", "> /dev/sda", "del", "rm", "rm -f", "chmod"
]

def check_deny_list(command: str) -> str | None:
    for pattern in DENY_LIST:
        if pattern in command:
            return f"Blocked: '{pattern}' is on the deny list"
    return None

PERMISSION_RULES = [
    {
        "tools": ["run_write", "run_edit"],
        "check": lambda args: not (WORKDIR / args.get("path", "")).resolve().is_relative_to(WORKDIR),
        "message": "Writing outside workspace",
    },
    {
        "tools": ["run_bash"],
        "check": lambda args: any(kw in args.get("command", "") for kw in ["rm ", "> /etc/", "chmod 777"]),
        "message": "Potentially destructive command",
    },
]

def check_rules(tool_name: str, args: dict) -> str | None:
    for rule in PERMISSION_RULES:
        if tool_name in rule["tools"] and rule["check"](args):
            return rule["message"]
    return None

class State(TypedDict):
    messages: Annotated[list, add_messages]
    max_iterations: int


In [3]:
HOOKS = {
    "UserPromptSubmit": [],
    "PreToolUse": [],
    "PostToolUse": [],
    "Stop": [],
}

def register_hook(event: str, callback):
    HOOKS[event].append(callback)

def trigger_hooks(event: str, *args):
    for callback in HOOKS[event]:
        result = callback(*args)
        if result is not None:   
            return result
    return None


def hard_deny_hook(tool_call: dict) -> str | None:
    """Hook to check hard deny lists (e.g., restricted bash commands)."""
    t_name = tool_call["name"]
    t_args = tool_call["args"]
    
    if t_name == "run_bash":
        deny_reason = check_deny_list(t_args.get("command", ""))
        if deny_reason:
            return f"{deny_reason}"
    return None


def conditional_security_hook(tool_call: dict) -> str | None:
    """Hook to evaluate rules and request Human-in-the-loop validation."""
    t_name = tool_call["name"]
    t_args = tool_call["args"]
    
    rule_reason = check_rules(t_name, t_args)
    if rule_reason:
        human_response = interrupt({
            "reason": rule_reason,
            "tool": t_name,
            "args": t_args
        })
        
        if human_response.get("action") != "allow":
            return "Permission denied by user."
            
    return None

register_hook("PreToolUse", hard_deny_hook)
register_hook("PreToolUse", conditional_security_hook)

In [4]:
@tool
def run_bash(command: str) -> str:
    """Run a bash command and return the output."""
    try:
        result = subprocess.run(command, shell=True, cwd=os.getcwd(), text=True, capture_output=True, timeout=120)
        out = (result.stdout + result.stderr).strip()
        return out[:10000] if out else "(No output)"
    except Exception as e:
        return f"Error: {str(e)}"

@tool 
def run_read(path: str, limit: int = None) -> str:
    """Read the contents of a file."""
    resolved = (WORKDIR / path).resolve()
    if not resolved.is_relative_to(WORKDIR):
        return "Error: Path escapes workspace"
    lines = resolved.read_text().splitlines()
    if limit:
        lines = lines[:limit]
    return "\n".join(lines)

@tool
def run_write(path: str, content: str) -> str:
    """Write content to a file."""
    resolved = (WORKDIR / path).resolve()
    if not resolved.is_relative_to(WORKDIR):
        return "Error: Path escapes workspace"
    resolved.write_text(content)
    return f"Wrote {len(content)} bytes to {path}"

@tool
def run_edit(path: str, old_text: str, new_text: str) -> str:
    """Edit a file by replacing the first occurrence of old_text with new_text."""
    resolved = (WORKDIR / path).resolve()
    if not resolved.is_relative_to(WORKDIR):
        return "Error: Path escapes workspace"
    text = resolved.read_text()
    if old_text not in text:
        return "Error: text not found"
    resolved.write_text(text.replace(old_text, new_text, 1))
    return f"Edited {path}"

TOOL_MAP = {
    "run_bash": run_bash, 
    "run_read": run_read, 
    "run_write": run_write, 
    "run_edit": run_edit
}
tools = list(TOOL_MAP.values())
tool_node = ToolNode(tools)

In [5]:
def call_model(state: State) -> Dict:
    """Node responsible solely for querying the LLM."""
    if state.get("max_iterations", 0) <= 0:
        return {"messages": [AIMessage(content="Reached maximum iterations. Stopping.")]}
        
    model = ChatOllama(model="gemma4", temperature=0.1)
    model_with_tools = model.bind_tools(tools)
    
    response = model_with_tools.invoke(state["messages"])
    return {
        "messages": [response],
        "max_iterations": state.get("max_iterations", 0) - 1
    }

def verify_and_route(state: State) -> Command[Literal["tools", "call_model", "__end__"]]:
    """Graph node executing validation logic before routing."""
    last_message = state["messages"][-1]    
    if not hasattr(last_message, "tool_calls") or not last_message.tool_calls:
        return Command(goto=END)
        
    if "stop" in last_message.content.lower():
        return Command(goto=END)

    for tool_call in last_message.tool_calls:
        blocked_reason = trigger_hooks("PreToolUse", tool_call)
        if blocked_reason:
            return Command(goto="call_model", update={"messages": [ToolMessage(content=blocked_reason, tool_call_id=tool_call["id"])]})

    return Command(goto="tools")

In [6]:
builder = StateGraph(State)

builder.add_node("call_model", call_model)
builder.add_node("verify_node", verify_and_route) 
builder.add_node("tools", tool_node)

builder.add_edge(START, "call_model")
builder.add_edge("call_model", "verify_node")    
builder.add_edge("tools", "call_model")

graph = builder.compile(checkpointer=MemorySaver())

initial_state = {
    "messages": [
        {"role": "user", "content": 'Create a file called test.txt and then write "Hello World" in it. Then read the contents of test.txt and write code to print them in python.'}
    ],
    "max_iterations": MAX_ITER
}

config = {"configurable": {"thread_id": "session_123"}}


result2 = graph.invoke(initial_state, config=config)
print("Final State:", result2)

Final State: {'messages': [HumanMessage(content='Create a file called test.txt and then write "Hello World" in it. Then read the contents of test.txt and write code to print them in python.', additional_kwargs={}, response_metadata={}, id='2e439730-be16-492f-8059-b46089c63d12'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4', 'created_at': '2026-05-25T16:51:29.0824168Z', 'done': True, 'done_reason': 'stop', 'total_duration': 163370147300, 'load_duration': 31348884400, 'prompt_eval_count': 289, 'prompt_eval_duration': 52403497700, 'eval_count': 422, 'eval_duration': 78154587100, 'logprobs': None, 'model_name': 'gemma4', 'model_provider': 'ollama'}, id='lc_run--019e6009-f489-7293-8dc2-6b447755e53e-0', tool_calls=[{'name': 'run_write', 'args': {'content': 'Hello World', 'path': 'test.txt'}, 'id': '880dab14-8350-4139-85cb-6f5145f79771', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 289, 'output_tokens': 422, 'total_tokens': 